# Notebook 07 — Violation Event Generation & Review Preparation

**Objective.** Turn the results of the earlier steps into one kind of record that a person can review, and make sure that nothing becomes a confirmed violation without that person.

**Background.** The rule, the tracker and the plate reader each produce their own output. A reviewer needs them joined: what happened, when, which vehicle, why the rule flagged it, what could not be settled, and images to judge by. Everything here stays a **candidate** until someone decides.

**Input.**
- `outputs/05b_red_light_violation/rule_v2/` and `outputs/06_license_plate_recognition/violation_clips/plates.json` — the candidates of the real footage, with their plates.
- `outputs/06_license_plate_recognition/simulated_red_light/` — the candidates of the simulated signal, with their plates.
- `outputs/05_violation_detection/time_reversed_control/` — wrong-way candidates on traffic played backward. They are not real, and only show that a second violation type fits the same record.
- `data/annotations/rlr_occasions_labels.json` — used here to script the simulated decisions.

**Output**, in `outputs/07_violation_event_generation/` (git-ignored):
- `<video name>/<video name>_review_events.json` — one validated record per occurrence, all `pending`.
- `<video name>/evidence/` — three images per event.
- `<video name>/<video name>_output.mp4` — all events in turn, for review.
- `simulated_review.json` — the scripted decisions, kept apart from the records.

The schema is documented in `docs/event_schema.md`.

**Plate numbers stay out of git.** A git filter removes every image output from the committed copy of this notebook, and printed text shows only a plate's first two characters and its length.

## 0. Setup

In [1]:
import copy
import json
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from traffic_violation_detection.events import (
    APPROVED,
    EVENT_SCHEMA_VERSION,
    PENDING,
    REJECTED,
    REQUIRED_FIELDS,
    SqliteViolationStore,
    confirmed_record,
    decide,
    deduplicate_records,
    final_plate,
    generate_review_events,
    load_records,
    plate_caption,
    rejection_record,
    validate_record,
)
from traffic_violation_detection.plates import READ, mask_plate
from traffic_violation_detection.red_light import load_red_light_scene, score_candidates
from traffic_violation_detection.video import iter_frames, read_frame_at, read_video_metadata

%config InlineBackend.figure_formats = ["jpeg"]

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"No pyproject.toml found in {start} or any parent directory")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUT_DIR = OUTPUTS / "07_violation_event_generation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SIMULATED_DIR = OUTPUTS / "06_license_plate_recognition" / "simulated_red_light"

SOURCES = {
    "rlr_occasions": (PROJECT_ROOT / "data" / "raw" / "rlr_occasions.mp4", OUTPUTS / "05b_red_light_violation" / "rule_v2", OUTPUTS / "06_license_plate_recognition" / "violation_clips" / "plates.json"),
    "traffic_video_modified": (PROJECT_ROOT / "data" / "traffic_video_modified.mp4", SIMULATED_DIR, SIMULATED_DIR / "plates" / "plates.json"),
}


def show(image, title=None, dpi=100):
    height, width = image.shape[:2]
    figure, axis = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.axis("off")
    if title:
        axis.set_title(title, fontsize=11, loc="left")
    plt.show()


def plate_text(record):
    # Only for printing: the status and a masked text, since this text is committed.
    plate = record["license_plate"]
    return f"{plate['status']} {mask_plate(plate['text']) or ''}".strip()

## 1. The event record

One structure for every violation type. Each field is required, and those that do not apply to a rule are null rather than missing. `docs/event_schema.md` explains each one.

In [3]:
print(f"schema version {EVENT_SCHEMA_VERSION}")
for field, kind in REQUIRED_FIELDS.items():
    names = " or ".join(k.__name__ for k in (kind if isinstance(kind, tuple) else (kind,)))
    print(f"  {field:<18}{names}")

schema version 1
  schema_version    int
  event_id          str
  video             dict
  violation_type    str
  track             dict
  time              dict
  rule              dict
  why               str
  signal_source     str or NoneType
  measurements      dict
  evidence          dict
  license_plate     dict
  needs_attention   list
  review            dict


## 2. From the outputs of the earlier steps to records

`generate_review_events` joins the small functions of `src/traffic_violation_detection/events.py`:

| Function | Task |
|---|---|
| `read_violation_outputs`, `read_plate_outputs` | Read what the earlier steps saved |
| `build_record` | Join the rule, track, times and plate into one record |
| `plate_summary`, `attention_flags` | Say how sure the plate is, and what the reviewer should check |
| `attach_evidence`, `write_evidence_frames` | Save three images and refer to the clip and plate crop |
| `deduplicate_records`, `same_occurrence` | Keep one record per occurrence |
| `validate_record` | Refuse a record that is not fit for review |
| `save_records` | Save the records |
| `write_review_video`, `draw_review_frame` | Save one video of all events in turn |
| `generate_review_events` | **Joins them all** |

In [4]:
runs = {}
for video_id, (video, violation_folder, plates_file) in SOURCES.items():
    runs[video_id] = generate_review_events(video, OUTPUT_DIR / video_id, violation_folder, plates_file)

for video_id, run in runs.items():
    records = run["records"]
    print(f"\n{video_id}: {len(records)} records, {len(run['duplicates'])} duplicates dropped")
    print(f"  {'event':<46}{'time s':>7}  {'plate':<16}needs attention")
    for record in records:
        print(f"  {record['event_id']:<46}{record['time']['trigger_time_s']:>7.2f}  {plate_text(record):<16}{', '.join(record['needs_attention']) or '-'}")


rlr_occasions: 17 records, 0 duplicates dropped
  event                                          time s  plate           needs attention
  evt-rlr_occasions-A-red_light-f00153-t12         5.10  unreadable      plate_unreadable
  evt-rlr_occasions-B-red_light-f00718-t23        23.93  unreadable      entry_not_seen, plate_unreadable
  evt-rlr_occasions-C-red_light-f01027-t1         34.23  no_plate_found  plate_not_found
  evt-rlr_occasions-C-red_light-f01059-t2         35.30  unreadable      plate_unreadable
  evt-rlr_occasions-D-red_light-f01166-t12        38.87  unreadable      plate_unreadable
  evt-rlr_occasions-E-red_light-f01588-t0         52.93  unreadable      plate_unreadable
  evt-rlr_occasions-F-red_light-f01761-t2         58.70  unreadable      plate_unreadable
  evt-rlr_occasions-G-red_light-f02033-t1         67.77  unreadable      plate_unreadable
  evt-rlr_occasions-J-red_light-f03248-t0        108.27  no_plate_found  plate_not_found
  evt-rlr_occasions-K-red_light-f03587

**A second violation type fits the same record.** Traffic played backward gives wrong-way candidates from Notebook 05. They are not real and stay in their own folder. Plate reading did not run on them, and the records say so.

In [5]:
control_folder = OUTPUTS / "05_violation_detection" / "time_reversed_control"
control = generate_review_events(control_folder / "video-test-reversed.mp4", OUTPUT_DIR / "wrong_way_control", control_folder)
print(f"wrong-way control: {len(control['records'])} records, {len(control['duplicates'])} duplicates dropped")
print("plate statuses:", dict(Counter(r["license_plate"]["status"] for r in control["records"])))
example = control["records"][0]
print(json.dumps({k: example[k] for k in ("event_id", "violation_type", "time", "why")}, indent=1))

wrong-way control: 11 records, 0 duplicates dropped
plate statuses: {'not_attempted': 11}
{
 "event_id": "evt-video-test-reversed-wrong_way-f00000-t12",
 "violation_type": "wrong_way",
 "time": {
  "start_frame": 0,
  "trigger_frame": 120,
  "end_frame": 120,
  "start_time_s": 0.0,
  "trigger_time_s": 2.002,
  "end_time_s": 2.002
 },
 "why": "Moved against the allowed direction of right_carriageway by 2.10 box heights in about 2.0 s, 31 degrees from the opposite direction."
}


## 3. What a reviewer gets

One event in full. The record states why the rule flagged it, and the images and clip let a person check. Boxes are drawn on the images only: the clip keeps its original pixels.

In [6]:
simulated = runs["traffic_video_modified"]["records"]
record = next(r for r in simulated if r["license_plate"]["status"] == READ)
shown = copy.deepcopy(record)
shown["license_plate"]["text"] = mask_plate(shown["license_plate"]["text"])  # The file itself keeps the full text.
print(json.dumps(shown, indent=1))

folder = OUTPUT_DIR / "traffic_video_modified"
tiles = []
for r in simulated:
    images = [cv2.imread(str(folder / f["image"])) for f in r["evidence"]["frames"]]
    row = np.hstack([cv2.resize(i, (480, round(i.shape[0] * 480 / i.shape[1])), interpolation=cv2.INTER_AREA) for i in images])
    tiles.append(row)
show(np.vstack(tiles), "Before, at and after the entry, for each simulated event. Local copy only.")

{
 "schema_version": 1,
 "event_id": "evt-traffic_video_modified-main-red_light-f00317-t23",
 "video": {
  "video_id": "traffic_video_modified",
  "fps": 26.333,
  "width": 1920,
  "height": 1012
 },
 "violation_type": "red_light",
 "track": {
  "tracking_session": "traffic_video_modified/ocsort-byte",
  "clip": "main",
  "track_id": "23",
  "class_name": "car"
 },
 "time": {
  "start_frame": null,
  "trigger_frame": 317,
  "end_frame": null,
  "start_time_s": null,
  "trigger_time_s": 12.038,
  "end_time_s": null
 },
 "rule": {
  "name": "red_light",
  "version": 2
 },
 "why": "Entered in lane 'through' while the through signal was red (1.0 s after it was first seen in that state), and continued past the stop line.",
 "signal_source": "simulated_overlay",
 "measurements": {
  "x": 1481.6,
  "lane": "through",
  "signal_group": "through",
  "signal_state": "red",
  "movement": "straight",
  "entry_seen": true,
  "entry_estimated": false,
  "box_xyxy": [
   1285.8,
   608.1,
   1675.7,


## 4. Validation

**Do the records hold?** Every saved record is checked again, and read back from its file. Each reference must lead to a file.

In [7]:
for video_id, run in runs.items():
    records = load_records(run["records_file"])
    assert records == json.loads(json.dumps(run["records"]))
    assert all(validate_record(r) == [] for r in records)
    assert all(r["review"]["status"] == PENDING and r["review"]["reviewer"] is None for r in records)
    assert len({r["event_id"] for r in records}) == len(records)
    folder = OUTPUT_DIR / video_id
    for r in records:
        paths = [r["evidence"]["clip"], *[f["image"] for f in r["evidence"]["frames"]]] + ([r["evidence"]["plate_crop"]] if r["evidence"]["plate_crop"] else [])
        assert all((folder / path).is_file() for path in paths), r["event_id"]
    print(f"{video_id}: {len(records)} records valid, all pending, all evidence present; statuses of plates: {dict(Counter(r['license_plate']['status'] for r in records))}")

rlr_occasions: 17 records valid, all pending, all evidence present; statuses of plates: {'unreadable': 14, 'no_plate_found': 3}
traffic_video_modified: 2 records valid, all pending, all evidence present; statuses of plates: {'read': 2}


**Is a broken record caught?** Each case damages a sound record in one way.

In [8]:
def damaged(change):
    record = copy.deepcopy(simulated[0])
    change(record)
    return validate_record(record)


CASES = {
    "no rule": lambda r: r.pop("rule"),
    "event id equal to the track id": lambda r: r.update(event_id=r["track"]["track_id"]),
    "event id of another video": lambda r: r.update(event_id="evt-other-1"),
    "track from another video": lambda r: r["track"].update(tracking_session="other/ocsort-byte"),
    "no trigger time": lambda r: r["time"].update(trigger_frame=None),
    "unknown plate status": lambda r: r["license_plate"].update(status="maybe"),
    "text on an unreadable plate": lambda r: r["license_plate"].update(status="unreadable"),
    "made-up review status": lambda r: r["review"].update(status="confirmed"),
    "approved with no reviewer": lambda r: r["review"].update(status=APPROVED),
    "pending with a reviewer": lambda r: r["review"].update(reviewer="someone"),
}
for name, change in CASES.items():
    problems = damaged(change)
    print(f"  {name:<34}{problems[0] if problems else 'NOT CAUGHT'}")
    assert problems

  no rule                           rule is missing
  event id equal to the track id    event_id must differ from the track id
  event id of another video         event_id must name its video
  track from another video          the track must be scoped to a tracking session of this video
  no trigger time                   the trigger time is invalid
  unknown plate status              the plate status is unknown
  text on an unreadable plate       the plate text does not fit its status
  made-up review status             the review status is unknown
  approved with no reviewer         a decision needs a reviewer
  pending with a reviewer           a pending record cannot have a reviewer


**Are duplicates prevented?** The earlier steps give no duplicates here, so these cases are built from real records. Only the first two should merge.

In [9]:
first = runs["traffic_video_modified"]["records"][0]
other = runs["traffic_video_modified"]["records"][1]


def variant(track_id, seconds_later, text=None, status=None):
    record = copy.deepcopy(first)
    record["event_id"] = f"{first['event_id']}-copy-{track_id}-{seconds_later}"
    record["track"]["track_id"] = track_id
    record["time"]["trigger_time_s"] += seconds_later
    record["license_plate"].update({k: v for k, v in (("text", text), ("status", status)) if v})
    return record


CASES = {
    "the same record, from a second run": copy.deepcopy(first),
    "the same plate on a new track, 2 s later": variant("99", 2.0),
    "the same plate, 60 s later": variant("99", 60.0),
    "another plate, 2 s later": variant("99", 2.0, text="ZZ0000"),
    "a suggested plate, 2 s later": variant("99", 2.0, status="uncertain"),
}
for name, repeat in CASES.items():
    kept, dropped = deduplicate_records([first, repeat])
    print(f"  {name:<44}{'merged' if dropped else 'kept apart'}")
kept, dropped = deduplicate_records([first, other])
print(f"  {'the two real events of the simulation':<44}{'merged' if dropped else 'kept apart'}")
assert not dropped

  the same record, from a second run          merged
  the same plate on a new track, 2 s later    merged
  the same plate, 60 s later                  kept apart
  another plate, 2 s later                    kept apart
  a suggested plate, 2 s later                kept apart
  the two real events of the simulation       kept apart


## 5. Simulated review

**These decisions come from a script, not from a person.** They exercise the contract, and confirm nothing. The script follows the labels of Notebook 05b, which no person has reviewed either: a candidate matching a `violation` label is approved, and one matching an `uncertain` label is rejected, since its signal at entry cannot be seen. The simulated events are approved, and one of them shows a reviewer correcting the plate.

The decisions go to a file of their own, and the records stay `pending`.

In [10]:
REVIEWER = "simulated-reviewer"
DECIDED_AT = "2026-11-02T12:00:00+00:00"
labels = json.loads((PROJECT_ROOT / "data" / "annotations" / "rlr_occasions_labels.json").read_text())
all_labels = {**labels["clips"], **labels["development_clips"]}
limits = labels["matching"]["max_frame_difference"], labels["matching"]["max_x_difference"]


def label_of(record):
    # The label matched to a candidate of the real footage, by the criterion of Notebook 05b.
    candidate = {"frame": record["time"]["trigger_frame"], "x": record["measurements"]["x"]}
    outcome = score_candidates([candidate], all_labels[record["track"]["clip"]]["events"], *limits)
    matched = outcome["true_positives"] + outcome["ignored"] + outcome["false_positives"]
    return matched[0][1] if matched else None


decided = []
for record in runs["rlr_occasions"]["records"]:
    label = label_of(record)
    if label is not None and label["label"] == "violation":
        decided.append(decide(record, APPROVED, REVIEWER, DECIDED_AT, notes="Scripted from a label."))
    else:
        decided.append(decide(record, REJECTED, REVIEWER, DECIDED_AT, notes="Scripted: the signal at entry cannot be seen."))
for index, record in enumerate(runs["traffic_video_modified"]["records"]):
    correction = record["license_plate"]["text"] if index == 1 else None  # A reviewer may also confirm a text by entering it.
    decided.append(decide(record, APPROVED, REVIEWER, DECIDED_AT, corrected_plate=correction, notes="Scripted, on a simulated signal."))

(OUTPUT_DIR / "simulated_review.json").write_text(json.dumps(decided, indent=1))
print("decisions:", dict(Counter(r["review"]["status"] for r in decided)))
assert all(validate_record(r) == [] for r in decided)
assert all(r["review"]["status"] == PENDING for run in runs.values() for r in load_records(run["records_file"]))
print("the records on file are all still pending")

for name, attempt in {
    "deciding twice": lambda: decide(decided[0], REJECTED, "someone else", DECIDED_AT),
    "deciding with no reviewer": lambda: decide(runs["rlr_occasions"]["records"][0], APPROVED, "", DECIDED_AT),
    "a status that does not exist": lambda: decide(runs["rlr_occasions"]["records"][0], "confirmed", REVIEWER, DECIDED_AT),
}.items():
    try:
        attempt()
        print(f"  {name}: NOT REFUSED")
    except ValueError as error:
        print(f"  {name}: refused ({error})")

decisions: {'approved': 18, 'rejected': 1}
the records on file are all still pending
  deciding twice: refused (evt-rlr_occasions-A-red_light-f00153-t12 was already decided: approved)
  deciding with no reviewer: refused (A decision needs a reviewer)
  a status that does not exist: refused (Unknown decision: confirmed)


## 6. What is persisted after review

Not everything in a record should outlive review:

- **Approved:** the facts of the violation, who decided and when, and references to its evidence. Tracks, per-frame reads and the rule's measurements stay behind.
- **Rejected:** only the decision, for audit. The plate and the images are dropped, because that vehicle was cleared.
- **Pending:** nothing.

A plate is kept only if a reviewer entered it, or if the OCR's status is `read`. A text that is merely `uncertain` is never kept by itself.

In [11]:
approved = [r for r in decided if r["review"]["status"] == APPROVED]
rejected = [r for r in decided if r["review"]["status"] == REJECTED]
kept = [confirmed_record(r) for r in approved]
audit = [rejection_record(r) for r in rejected]
print(f"{len(kept)} confirmed records, {len(audit)} audit records")
print("confirmed record fields:", ", ".join(kept[0]))
print("audit record fields:    ", ", ".join(audit[0]))
print("plate kept by source:", dict(Counter(str(k["plate_source"]) for k in kept)))
assert not any("plate" in key or "evidence" in key for a in audit for key in a)

for name, record in (("a pending record", runs["rlr_occasions"]["records"][0]), ("a rejected record", rejected[0])):
    try:
        confirmed_record(record)
        print(f"  persisting {name}: NOT REFUSED")
    except ValueError as error:
        print(f"  persisting {name}: refused")

18 confirmed records, 1 audit records
confirmed record fields: event_id, video_id, violation_type, trigger_time_s, vehicle_class, plate_text, plate_source, signal_source, rule_name, rule_version, evidence, reviewer, decided_at, notes
audit record fields:     event_id, video_id, violation_type, rule_name, rule_version, reviewer, decided_at, notes
plate kept by source: {'None': 16, 'ocr': 1, 'reviewer': 1}
  persisting a pending record: refused
  persisting a rejected record: refused


## 7. The database interface

`ViolationStore` is the interface the application will implement. `SqliteViolationStore` is a small implementation that tests it, here in memory. The real database belongs to the application, and is not built here.

In [12]:
store = SqliteViolationStore()
for record in kept:
    store.save(record)
print("stored:", store.count())

sample = next(k for k in kept if k["plate_text"])
assert store.get(sample["event_id"]) == sample
assert [r["event_id"] for r in store.find_by_plate(sample["plate_text"])] == [sample["event_id"]]
print(f"found by plate {mask_plate(sample['plate_text'])}: 1 violation, event {sample['event_id']}")

for name, attempt in {
    "the same event again": lambda: store.save(sample),
    "a candidate that skipped review": lambda: store.save(runs["rlr_occasions"]["records"][0]),
    "a confirmed record with no reviewer": lambda: store.save({**sample, "event_id": "evt-x", "reviewer": None}),
}.items():
    try:
        attempt()
        print(f"  {name}: NOT REFUSED")
    except ValueError:
        print(f"  {name}: refused")
print("still stored:", store.count())
assert store.count() == len(kept)

stored: 18
found by plate YB****: 1 violation, event evt-traffic_video_modified-main-red_light-f00317-t23
  the same event again: refused
  a candidate that skipped review: refused
  a confirmed record with no reviewer: refused
still stored: 18


## 8. The video for review

Each folder has one video of all its events in turn, with each event's plate result and status on it.

In [13]:
for video_id, run in runs.items():
    meta = read_video_metadata(run["output_video"])
    frames = sum(1 for _ in iter_frames(run["output_video"]))
    clips = json.loads((SOURCES[video_id][1] / f"{video_id}_violation_clips" / "manifest.json").read_text())["clips"]
    print(f"{run['output_video'].relative_to(PROJECT_ROOT)}: {meta['width']}x{meta['height']}, {meta['fps']:.2f} FPS, {frames} frames, {frames / meta['fps']:.1f} s, {run['output_video'].stat().st_size / 1e6:.1f} MB")
    assert frames == sum(c["frame_count"] for c in clips)
frame = read_frame_at(runs["traffic_video_modified"]["output_video"], 90)["image"]
show(cv2.resize(frame, (900, round(frame.shape[0] * 900 / frame.shape[1])), interpolation=cv2.INTER_AREA), "A frame of the review video. Local copy only.")

outputs/07_violation_event_generation/rlr_occasions/rlr_occasions_output.mp4: 1280x720, 30.00 FPS, 2336 frames, 77.9 s, 19.6 MB


outputs/07_violation_event_generation/traffic_video_modified/traffic_video_modified_output.mp4: 1920x1012, 26.33 FPS, 266 frames, 10.1 s, 10.7 MB


## Results

All numbers come from the outputs above.

- **Records.** 19 candidates became 19 valid records: 17 from the real footage, with rule version 2, and 2 from the simulated signal. All are `pending`, and every reference leads to a file.
- **What a reviewer is told.** Every record that cannot be settled by itself says so in `needs_attention`. On the real footage no plate could be read, as 14 records state `plate_unreadable` and 3 `plate_not_found`. 2 records state `entry_not_seen`. Both simulated records state `simulated_signal`, and one also states `entry_estimated`.
- **A second violation type.** 11 wrong-way records from the backward control fit the same structure, with `plate_not_attempted`.
- **Validation.** All 10 damaged records were caught. Duplicates merged in the 2 cases that should merge and stayed apart in the other 4.
- **Simulated review.** 18 approved and 1 rejected, from a script. Deciding twice, deciding with no reviewer and using a status that does not exist were all refused, and the records on file stayed `pending`.
- **Persistence.** Of the 18 confirmed records, 1 keeps the OCR's plate, 1 a plate entered by the reviewer, and 16 have none. The rejected record keeps no plate or evidence. A pending record and a rejected record could not be persisted, and the store refused a repeat, a candidate that skipped review, and a record with no reviewer.

**A mistake caught by looking at the images.** For a vehicle hidden at its entry, the first version of the evidence showed the moment of the estimated entry, when another car filled the frame and the flagged car could not be seen. A reviewer could have taken the other car for the offender. Such a record now shows the estimated entry, the moment the vehicle is first seen, and after that.

## Limitations

- **No person has made a decision.** The review is scripted from labels that no person has reviewed, so the 18 "confirmed" records only test the contract. None of them is a finding about a driver.
- **Most confirmed records have no plate.** 16 of 18 could not be linked to a vehicle, because the real footage is too blurred. A real system should decide what happens to such a case.
- **Duplicates with no plate.** Two tracks of one vehicle with unreadable plates would stay two records, because nothing identifies them as one. A reviewer would have to see this.
- **No real case could be filed from these records.** They hold the time within a video, and not a camera's identity, its location, or the date and time. Test videos do not provide these.
- **A decision cannot be changed.** A real application needs a logged way to do that.
- **Deleting evidence is left to the application.** Rejection keeps no plate in its audit record, but the image files stay on disk until something removes them.
- **The evidence of estimated entries is weaker.** The crossing itself was not seen, and the estimate was about 20 frames from the moment seen by eye in the one real case available.
- **Storage is only a test.** `SqliteViolationStore` has no access control, retention or backup.

## Conclusion

**Completion criterion** — *each candidate event contains sufficient information for later review, and no candidate is automatically treated as a confirmed violation*:

- **Sufficient information: met for review, not for filing a case.** Each record has the reason, the times, the vehicle, the plate result, the rule's version, what needs attention, a clip and three images.
- **No automatic confirmation: met.** `decide` is the only way out of `pending`, it needs a named reviewer, and the store takes nothing else.

**Readiness for integration.** The schema is in `docs/event_schema.md` and the code in `src/traffic_violation_detection/events.py`, with tests. A review application can read `<video name>_review_events.json`, and implement `ViolationStore`.

**Next steps.**

- **Notebook 08 — end-to-end baseline** can join the steps into one call, from a video to records ready for review.
- **A person's review** of the labels remains the most valuable check on every score so far.